In [50]:
import os
import time
from unstract.llmwhisperer import LLMWhispererClientV2

from dotenv import load_dotenv
load_dotenv()

client = LLMWhispererClientV2(base_url='https://llmwhisperer-api.eu-west.unstract.com/api/v2',
                              api_key=os.getenv('API_KEY'))

pdf_file_path = 'DIMO.pdf'
pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
result = client.whisper(file_path=pdf_file_path)

while True:
    status = client.whisper_status(whisper_hash=result['whisper_hash'])
    if status['status'] == 'processed':
        resultx = client.whisper_retrieve(whisper_hash=result['whisper_hash'])
        break

    time.sleep(5)

extracted_table = resultx['extraction']['result_text']

2026-07-17 22:56:10,218 - unstract.llmwhisperer.client_v2 - DEBUG - logging_level set to DEBUG
2026-07-17 22:56:10,220 - unstract.llmwhisperer.client_v2 - DEBUG - base_url set to https://llmwhisperer-api.eu-west.unstract.com/api/v2
2026-07-17 22:56:10,222 - unstract.llmwhisperer.client_v2 - DEBUG - whisper called
2026-07-17 22:56:10,223 - unstract.llmwhisperer.client_v2 - DEBUG - api_url: https://llmwhisperer-api.eu-west.unstract.com/api/v2/whisper
2026-07-17 22:56:10,225 - unstract.llmwhisperer.client_v2 - DEBUG - params: {'mode': 'form', 'output_mode': 'layout_preserving', 'page_seperator': '<<<', 'pages_to_extract': '', 'median_filter_size': 0, 'gaussian_blur_radius': 0, 'line_splitter_tolerance': 0.4, 'horizontal_stretch_factor': 1.0, 'mark_vertical_lines': False, 'mark_horizontal_lines': False, 'line_spitter_strategy': 'left-priority', 'add_line_nos': False, 'include_line_confidence': False, 'lang': 'eng', 'tag': 'default', 'filename': '', 'webhook_metadata': '', 'use_webhook': ''

In [51]:
print(extracted_table)



TEN YEAR                         SUMMARY 

 Year Ended 31st March                         2025/26     2024/25     2023/24      2022/23     2021/22     2020/21     2019/20      2018/19     2017/18     2016/17 
                                                Rs.'000     Rs.'000      Rs.'000     Rs.'000     Rs.'000     Rs.'000     Rs.'000      Rs.'000     Rs.'000     Rs.'000 

 Group 
 Operating Results 
 Revenue                                   103,633,939 50,174,979 43,644,295 35,299,201 37,507,480 30,819,014 34,557,871 38,300,350 43,686,158 44,492,990 
 Profit/(Loss) before taxation                2,046,625 (1,948,055)     183,111     737,256    1,165,345    720,111     279,527      104,119     716,607    1,043,392 
 Income tax expense/(reversal)                 (502,672)   639,589      (90,506)     (39,112)   (310,975)   (183,845)    (78,383)    (27,436)   (193,391)    (386,601) 
 Profit/(Loss) for the year                   1,543,953 (1,308,466)      92,605     698,144     854,370

In [52]:
import pandas as pd
import re

def lineextractor(extracted_table):
    lines = extracted_table.strip().split('\n')
    target_yrs = ["2025/26", "2024/25", "2023/24", "2022/23", "2021/22", "2020/21"]

    data_store= {year:{} for year in target_yrs}

    hooks = {
        "Revenue": "revenue",
        "Profit/(Loss) before taxation": "pbt",
        "Total equity": "total_equity",
        "Total borrowings": "total_borrowings",
        "Current assets": "current_assets",
        "Total assets employed": "total_assets",
        "Current ratio (No. of times)": "current_ratio",
        "Quick asset ratio": "quick_ratio",
        "Gearing ratio (%)": "gearing_ratio"
    }

    for line in lines:
        clean_line = " ".join(line.split())

        for text_label, json_key in hooks.items():

            if text_label == "Revenue" and "Revenue reserve" in clean_line:
                continue

            if text_label in clean_line:
                number_part = clean_line[len(text_label):].strip()
                token = number_part.split()

                clean_numbers = []
                for token in token:
                    is_negative = '(' in token 
                    cleaned = token.replace(',', '').replace('(', '').replace(')', '')

                    if cleaned.replace('.', '', 1).isdigit():
                        val = float(cleaned)
                        if is_negative:
                            val = -val
                        clean_numbers.append(val)


                for idx, year in enumerate(target_yrs):
                    if idx < len(clean_numbers):
                        data_store[year][json_key] = clean_numbers[idx]
    return data_store

extracted_dict = lineextractor(extracted_table)
print("Dictionary Built!")

Dictionary Built!


In [53]:
df = pd.DataFrame(extracted_dict)

df = df.T
df.index.name = 'Year'
df = df.reset_index()

df

,Year,revenue,pbt,total_equity,total_borrowings,current_assets,total_assets,gearing_ratio,current_ratio,quick_ratio
0,2025/26,103633939.0,2046625.0,16860840.0,47486374.0,55505127.0,101189532.0,74.00,1.04,0.54
1,2024/25,50174979.0,-1948055.0,15438343.0,27923186.0,33265547.0,70044261.0,64.00,1.01,0.69
2,2023/24,43644295.0,183111.0,16848635.0,22394791.0,28372573.0,58742570.0,57.00,1.16,0.76
3,2022/23,35299201.0,737256.0,15353631.0,13730034.0,22300228.0,50041109.0,47.00,1.16,0.64
4,2021/22,37507480.0,1165345.0,15466487.0,8951541.0,25075075.0,21503681.0,36.66,1.23,0.75
5,2020/21,30819014.0,720111.0,14961252.0,4551348.0,17521696.0,19512600.0,23.33,1.39,0.90


In [54]:
analysis_df = df.copy()
base_year_row = analysis_df.loc[5]  # 2021/22 acts as the baseline (100%)
core_metrics = ['revenue', 'pbt', 'total_equity', 'total_borrowings', 'current_assets', 'total_assets']

# Helper variables for liability and working capital metrics
current_liabs = analysis_df['current_assets'] / analysis_df['current_ratio']
analysis_df['net_working_capital_abs'] = analysis_df['current_assets'] - current_liabs

# 1. Horizontal Growth & Trend Index Analyses
for metric in core_metrics:
    analysis_df[f'{metric}_horizontal_growth_%'] = analysis_df[metric].pct_change(-1) * 100
    analysis_df[f'{metric}_trend_index'] = (analysis_df[metric] / base_year_row[metric]) * 100

# 2. Vertical Analysis (Common-Size Structure)
analysis_df['current_assets_vertical_%'] = (analysis_df['current_assets'] / analysis_df['total_assets']) * 100
analysis_df['total_equity_vertical_%'] = (analysis_df['total_equity'] / analysis_df['total_assets']) * 100
analysis_df['borrowings_vertical_%'] = (analysis_df['total_borrowings'] / analysis_df['total_assets']) * 100
analysis_df['pbt_vertical_margin_%'] = (analysis_df['pbt'] / analysis_df['revenue']) * 100

# 3. Categorized Financial Ratios
analysis_df['net_profit_margin_%'] = (analysis_df['pbt'] / analysis_df['revenue']) * 100
analysis_df['return_on_assets_roa_%'] = (analysis_df['pbt'] / analysis_df['total_assets']) * 100
analysis_df['return_on_equity_roe_%'] = (analysis_df['pbt'] / analysis_df['total_equity']) * 100
analysis_df['working_capital_to_assets_%'] = (analysis_df['net_working_capital_abs'] / analysis_df['total_assets']) * 100
analysis_df['debt_to_equity_ratio'] = analysis_df['total_borrowings'] / analysis_df['total_equity']
analysis_df['equity_to_total_assets_%'] = (analysis_df['total_equity'] / analysis_df['total_assets']) * 100

# 4. Altman Z-Score Parameterization & Calculation
analysis_df['Z_X1'] = analysis_df['net_working_capital_abs'] / analysis_df['total_assets']
analysis_df['Z_X2'] = analysis_df['total_equity'] / analysis_df['total_assets']
analysis_df['Z_X3'] = analysis_df['pbt'] / analysis_df['total_assets']
analysis_df['Z_X4'] = analysis_df['total_equity'] / analysis_df['total_borrowings']
analysis_df['Z_X5'] = analysis_df['revenue'] / analysis_df['total_assets']

analysis_df['altman_z_score'] = (1.2 * analysis_df['Z_X1']) + (1.4 * analysis_df['Z_X2']) + \
                               (3.3 * analysis_df['Z_X3']) + (0.6 * analysis_df['Z_X4']) + \
                               (0.999 * analysis_df['Z_X5'])

def categorize_z_zone(z):
    if z > 2.90: return "Safe Zone"
    elif z < 1.23: return "Distress Zone"
    return "Grey Zone"

analysis_df['altman_risk_classification'] = analysis_df['altman_z_score'].apply(categorize_z_zone)

# Final formatting clean up
analysis_df = analysis_df.round(2)

trimmed_analysis_df = analysis_df.iloc[:5].copy()
# ==============================================================================
# 5. SPLIT INTO TWO DISTINCT DATAFRAMES
# ==============================================================================
# Extract the clean, unparsed original metrics
original_cols = ['Year', 'revenue', 'pbt', 'total_equity', 'total_borrowings', 
                 'current_assets', 'total_assets', 'gearing_ratio', 'current_ratio', 'quick_ratio']
original_df = analysis_df[original_cols].copy()

# Extract the calculated trends, vertical analyses, ratios, and risk scores
analysis_cols = [col for col in analysis_df.columns if col not in original_cols or col == 'Year']
analysis_results_df = analysis_df[analysis_cols].iloc[:5].copy()

# Print out status verification
print(f"Data split successfully!\n- original_df shape: {original_df.shape}\n- analysis_results_df shape: {analysis_results_df.shape}")

Data split successfully!
- original_df shape: (6, 10)
- analysis_results_df shape: (5, 31)


In [55]:
original_df

,Year,revenue,pbt,total_equity,total_borrowings,current_assets,total_assets,gearing_ratio,current_ratio,quick_ratio
0,2025/26,103633939.0,2046625.0,16860840.0,47486374.0,55505127.0,101189532.0,74.00,1.04,0.54
1,2024/25,50174979.0,-1948055.0,15438343.0,27923186.0,33265547.0,70044261.0,64.00,1.01,0.69
2,2023/24,43644295.0,183111.0,16848635.0,22394791.0,28372573.0,58742570.0,57.00,1.16,0.76
3,2022/23,35299201.0,737256.0,15353631.0,13730034.0,22300228.0,50041109.0,47.00,1.16,0.64
4,2021/22,37507480.0,1165345.0,15466487.0,8951541.0,25075075.0,21503681.0,36.66,1.23,0.75
5,2020/21,30819014.0,720111.0,14961252.0,4551348.0,17521696.0,19512600.0,23.33,1.39,0.90


In [56]:
trimmed_analysis_df

,Year,revenue,pbt,total_equity,total_borrowings,current_assets,total_assets,gearing_ratio,current_ratio,quick_ratio,...,working_capital_to_assets_%,debt_to_equity_ratio,equity_to_total_assets_%,Z_X1,Z_X2,Z_X3,Z_X4,Z_X5,altman_z_score,altman_risk_classification
0,2025/26,103633939.0,2046625.0,16860840.0,47486374.0,55505127.0,101189532.0,74.00,1.04,0.54,...,2.11,2.82,16.66,0.02,0.17,0.02,0.36,1.02,1.56,Grey Zone
1,2024/25,50174979.0,-1948055.0,15438343.0,27923186.0,33265547.0,70044261.0,64.00,1.01,0.69,...,0.47,1.81,22.04,0.00,0.22,-0.03,0.55,0.72,1.27,Grey Zone
2,2023/24,43644295.0,183111.0,16848635.0,22394791.0,28372573.0,58742570.0,57.00,1.16,0.76,...,6.66,1.33,28.68,0.07,0.29,0.00,0.75,0.74,1.69,Grey Zone
3,2022/23,35299201.0,737256.0,15353631.0,13730034.0,22300228.0,50041109.0,47.00,1.16,0.64,...,6.15,0.89,30.68,0.06,0.31,0.01,1.12,0.71,1.93,Grey Zone
4,2021/22,37507480.0,1165345.0,15466487.0,8951541.0,25075075.0,21503681.0,36.66,1.23,0.75,...,21.80,0.58,71.92,0.22,0.72,0.05,1.73,1.74,4.23,Safe Zone


In [58]:
excel_name = f"{pdf_name}_analysis.xlsx"
folder_path = "output"

excel_path = os.path.join(folder_path, excel_name)

with pd.ExcelWriter(excel_path) as writer:
    original_df.to_excel(writer, sheet_name='Original Metrics', index=False)
    analysis_results_df.to_excel(writer, sheet_name='Analysis Results', index=False)

print(f"Excel file '{excel_name}' created successfully in the '{folder_path}' folder.")

Excel file 'DIMO_analysis.xlsx' created successfully in the 'output' folder.
